In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F

import numpy as np
import pandas as pd
import copy
import random
import zsx_some_tools as st

from torch.optim import Adam
from sklearn.model_selection import train_test_split
from utils import AverageMeter, ProgressMeter, get_state_dict, save_ckpt
from sklearn.metrics import confusion_matrix
from sklearn import metrics
import matplotlib.pyplot as plt

import time

In [2]:
# 创建自定义 Dataset 类
class CustomDataset(Dataset):
    def __init__(self, input_data, output_data):
        self.input_data = input_data
        self.output_data = output_data

    def __len__(self):
        return len(self.input_data)

    def __getitem__(self, idx):
        # 返回每个索引对应的输入和输出数据
        return torch.tensor(self.input_data[idx], dtype=torch.float32), torch.tensor(self.output_data[idx], dtype=torch.float32)

In [3]:
def get_dataset(data_x_, data_y_, batch_size=128, shuffle=True, drop_last=True, balance=True,
                train_val_test=(0.8, 0.1, 0.1), seed=42):

    if train_val_test[0] == 0 or (train_val_test[1] == 0 and train_val_test[2] > 0):
        raise ValueError('train_val_test is not suitable, please reset. e.g. (0.8, 0.1, 0.1)')
    
    def _produce_data(pos_, neg_):
        pos_.loc[:, 'label'] = 1
        neg_.loc[:, 'label'] = 0
        data_ = pd.concat([pos_, neg_])
        
        data_x_ = data_.iloc[:, :-1]
        data_y_ = list(data_.iloc[:, -1])
        
        data_x__ = torch.tensor(np.array(data_x_), dtype=torch.float32)
        data_y__ = torch.tensor(np.array(data_y_), dtype=torch.float32)
        _dataset = torch.utils.data.TensorDataset(data_x__, data_y__.long())
        
        return DataLoader(_dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last)

    # compute split ratio
    train_ratio = train_val_test[0]
    test_ratio = train_val_test[2]
    rest_ratio = 1 - train_ratio
    if test_ratio > 0:
        rest_ratio2 = test_ratio / rest_ratio
    else:
        rest_ratio2 = 0
        
    pos_data_x_ = data_x_.loc[data_y_ == 1]
    neg_data_x_ = data_x_.loc[data_y_ == 0]
    
    if balance:
        num1 = pos_data_x_.shape[0]
        num2 = neg_data_x_.shape[0]
        if num1 == num2:
            pass
        elif num1 > num2:
            sample_id = random.sample(list(range(num1)), num2)
            pos_data_x_ = pos_data_x_.iloc[sample_id]
        else:
            sample_id = random.sample(list(range(num2)), num1)
            neg_data_x_ = neg_data_x_.iloc[sample_id]

    # split dataset
    if train_ratio == 1:
        train_data_ = _produce_data(pos_data_x_, neg_data_x_)
        return train_data_, None, None

    else:
        pos_train_data_x, pos_test_data_x = train_test_split(pos_data_x_, test_size=rest_ratio, random_state=seed)
        neg_train_data_x, neg_test_data_x = train_test_split(neg_data_x_, test_size=rest_ratio, random_state=seed)
        if rest_ratio2:
            pos_val_data_x, pos_test_data_x = train_test_split(pos_test_data_x, test_size=rest_ratio2, random_state=seed)
            neg_val_data_x, neg_test_data_x = train_test_split(neg_test_data_x, test_size=rest_ratio2, random_state=seed)
            
            test_data_ = _produce_data(pos_test_data_x, neg_test_data_x)

        else:
            pos_val_data_x, neg_val_data_x = pos_test_data_x, neg_test_data_x
            test_data_ = None

        train_data_ = _produce_data(pos_train_data_x, neg_train_data_x)
        val_data_ = _produce_data(pos_val_data_x, neg_val_data_x)

        return train_data_, val_data_, test_data_

In [4]:
def get_dataset_remove(data_x_, data_y_, remove_list, batch_size=128, shuffle=True, drop_last=True, balance=True,
                train_val_test=(0.8, 0.1, 0.1), seed=42):

    if train_val_test[0] == 0 or (train_val_test[1] == 0 and train_val_test[2] > 0):
        raise ValueError('train_val_test is not suitable, please reset. e.g. (0.8, 0.1, 0.1)')
    
    def _produce_data(pos_, neg_):
        pos_.loc[:, 'label'] = 1
        neg_.loc[:, 'label'] = 0
        data_ = pd.concat([pos_, neg_])
        
        data_x_ = data_.iloc[:, :-1]
        data_y_ = list(data_.iloc[:, -1])
        
        data_x__ = torch.tensor(np.array(data_x_), dtype=torch.float32)
        data_y__ = torch.tensor(np.array(data_y_), dtype=torch.float32)
        _dataset = torch.utils.data.TensorDataset(data_x__, data_y__.long())
        
        return DataLoader(_dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last)

    # compute split ratio
    train_ratio = train_val_test[0]
    test_ratio = train_val_test[2]
    rest_ratio = 1 - train_ratio
    if test_ratio > 0:
        rest_ratio2 = test_ratio / rest_ratio
    else:
        rest_ratio2 = 0
        
    pos_data_x_ = data_x_.loc[data_y_ == 1]
    neg_data_x_ = data_x_.loc[data_y_ == 0]
    
    if balance:
        num1 = pos_data_x_.shape[0]
        num2 = neg_data_x_.shape[0]
        if num1 == num2:
            pass
        elif num1 > num2:
            sample_id = random.sample(list(range(num1)), num2)
            pos_data_x_ = pos_data_x_.iloc[sample_id]
        else:
            sample_id = random.sample(list(range(num2)), num1)
            neg_data_x_ = neg_data_x_.iloc[sample_id]

    # split dataset
    if train_ratio == 1:
        train_data_ = _produce_data(pos_data_x_, neg_data_x_)
        return train_data_, None, None

    else:
        pos_train_data_x, pos_test_data_x = train_test_split(pos_data_x_, test_size=rest_ratio, random_state=seed)
        neg_train_data_x, neg_test_data_x = train_test_split(neg_data_x_, test_size=rest_ratio, random_state=seed)
        if rest_ratio2:
            pos_val_data_x, pos_test_data_x = train_test_split(pos_test_data_x, test_size=rest_ratio2, random_state=seed)
            neg_val_data_x, neg_test_data_x = train_test_split(neg_test_data_x, test_size=rest_ratio2, random_state=seed)
            
            remove_set = set(remove_list)
            pos_index = list(set(pos_test_data_x.index) - remove_set)
            neg_index = list(set(neg_test_data_x.index) - remove_set)
            
            pos_test_data_x = pos_test_data_x.loc[pos_index]
            neg_test_data_x = neg_test_data_x.loc[neg_index]
            
            test_data_ = _produce_data(pos_test_data_x, neg_test_data_x)

        else:
            pos_val_data_x, neg_val_data_x = pos_test_data_x, neg_test_data_x
            test_data_ = None

        train_data_ = _produce_data(pos_train_data_x, neg_train_data_x)
        val_data_ = _produce_data(pos_val_data_x, neg_val_data_x)

        return train_data_, val_data_, test_data_

# BATT

In [5]:
class BernoulliStraightThrough(torch.autograd.Function):
    """伯努利采样的直通估计器"""

    @staticmethod
    def forward(ctx, probs):
        # 前向：离散化采样
        sample = torch.bernoulli(probs)
        ctx.save_for_backward(probs, sample)
        return sample

    @staticmethod
    def backward(ctx, grad_output):
        probs, sample = ctx.saved_tensors
        # 直通估计器：梯度直接传回probs，忽略采样操作
        grad_probs = grad_output.clone()
        return grad_probs

class BernoulliMultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=None, batch_first=None):
        super(BernoulliMultiHeadAttention, self).__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        assert self.head_dim * num_heads == embed_dim, "embed_dim must be divisible by num_heads"

        self.qkv_proj = nn.Linear(embed_dim, embed_dim * 3)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x, x1, x2):
        batch_size, seq_length, embed_dim = x.shape

        # Compute Q, K, V
        qkv = self.qkv_proj(x)  # (batch_size, seq_length, 3 * embed_dim)
        qkv = qkv.reshape(batch_size, seq_length, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)  # (3, batch_size, num_heads, seq_length, head_dim)
        q, k, v = qkv[0], qkv[1], qkv[2]

        # Compute scaled dot-product attention scores
        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)  # (batch_size, num_heads, seq_length, seq_length)

        # Apply sigmoid instead of softmax
        attn_probs = torch.sigmoid(attn_scores)
        
        # wrong
#         attn_probs = torch.bernoulli(attn_probs)  # 采样 {0,1}
        # STE
        attn_probs = BernoulliStraightThrough.apply(attn_probs)

        # Apply attention to V
        attn_output = torch.matmul(attn_probs, v)  # (batch_size, num_heads, seq_length, head_dim)
        attn_output = attn_output.permute(0, 2, 1, 3).reshape(batch_size, seq_length, embed_dim)

        return self.out_proj(attn_output), None

In [6]:
class CSA2(nn.Module):
    def __init__(self, in_dim=1280, h_dim=100, out_dim=2, dropout=0., num_heads=4):
        super(CSA2, self).__init__()
        self.in_dim = in_dim
        self.h_dim = h_dim
        self.out_dim = out_dim
        self.num_heads = num_heads

        # 线性投影层，用于将输入投影到 Self-Attention 所需的维度
        self.fc_input = nn.Linear(in_features=in_dim, out_features=h_dim, bias=True)

        # 第一层 多头自注意力 + LayerNorm
        self.multihead_attn_1 = BernoulliMultiHeadAttention(embed_dim=h_dim, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(h_dim)  # 第一层 LayerNorm

        # 第二层 多头自注意力 + LayerNorm
        self.multihead_attn_2 = BernoulliMultiHeadAttention(embed_dim=h_dim, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(h_dim)  # 第二层 LayerNorm

        # 前馈神经网络（FFN）
        self.ffn = nn.Sequential(
            nn.Linear(h_dim, h_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(h_dim, h_dim),
        )

        # 分类头
        self.fc_output = nn.Linear(h_dim, out_dim)

        # 其他组件
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # 线性投影
        x = self.fc_input(x)  # (batch_size, h_dim)
        x = x.unsqueeze(1)    # (batch_size, seq_len=1, h_dim)

        # 第一层 MultiheadAttention + LayerNorm + 残差连接
        attn_output1, _ = self.multihead_attn_1(x, x, x)  # (batch_size, seq_len, h_dim)
        x = self.norm1(x + attn_output1)  # 残差连接 + 归一化

        # 第二层 MultiheadAttention + LayerNorm + 残差连接
        attn_output2, _ = self.multihead_attn_2(x, x, x)  # (batch_size, seq_len, h_dim)
        x = self.norm2(x + attn_output2)  # 残差连接 + 归一化

        return x


In [7]:
def accuracy(output, target):
    return torch.mean((torch.argmax(output, dim=1) == target).float())

In [8]:
class GatingNetwork(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_experts):
        super(GatingNetwork, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, num_experts)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        logits = self.fc2(x)
        return F.softmax(logits, dim=-1)

class MixtureOfExperts(nn.Module):
    def __init__(self, Input_expert, input_dim, hidden_dim, output_dim, num_experts, expert_hidden_dim, 
                 out_dim, dropout):
        super(MixtureOfExperts, self).__init__()
        self.num_experts = num_experts
        
        # 创建多个专家
        self.experts = nn.ModuleList([Input_expert(input_dim, expert_hidden_dim, output_dim) for _ in range(num_experts)])
        
        # 创建门控网络
        self.gating_network = GatingNetwork(input_dim, hidden_dim, num_experts)
        
        # 前馈神经网络（FFN）
        self.ffn = nn.Sequential(
            nn.Linear(expert_hidden_dim, expert_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(expert_hidden_dim, expert_hidden_dim),
        )

        # 分类头
        self.fc_output = nn.Linear(expert_hidden_dim, out_dim)

        # 其他组件
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # 门控网络计算每个专家的权重
        gating_weights = self.gating_network(x)
        
        # 每个专家输出
        expert_outputs = torch.stack([expert(x) for expert in self.experts], dim=-1)
        
        # 加权平均专家的输出
        gating_weights = torch.softmax(gating_weights.unsqueeze(1), dim=-1)  # 对权重进行 softmax 归一化
        
        # 加权求和
        x = torch.sum(expert_outputs * gating_weights.unsqueeze(2), dim=-1)

        # 去掉时间维度
        x = x.squeeze(1)  # (batch_size, h_dim)

        # 前馈神经网络
        x = self.ffn(x)  # (batch_size, h_dim)
        x = self.relu(x)
        x = self.dropout(x)

        # 分类层
        x = self.fc_output(x)  # (batch_size, out_dim)
        x = self.softmax(x)    # 归一化为概率
        
        return x

In [9]:
def train(epoch, max_epoch, data_loader, model, criterion, optimizer, device, log_file=None):
    model.train()
    end = time.time()
    loss_total = []
    acc1_total = []
    for idx, (feature, target) in enumerate(data_loader):
        feature = feature.to(device)
        target = target.to(device)
        output = model(feature)

        loss = criterion(output, target)
        acc1 = accuracy(output, target)

        loss_total += [loss.item()]
        acc1_total += [acc1.item()]

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        end = time.time()
        if idx % 100 == 0:
            print('batch:', idx, '/', len(data_loader), end='\r')
    
    return np.mean(loss_total), np.mean(acc1_total)

In [10]:
def evaluate(data_loader, model, criterion, device):
    model.eval()
    
    output_all, target_all = None, None
    for idx, (feature, target) in enumerate(data_loader):
        feature = feature.to(device)
        target = target.to(device)
        output = model(feature)
        if idx == 0:
            output_all = output
            target_all = target
        else:
            output_all = torch.cat([output_all, output], 0)
            target_all = torch.cat([target_all, target], 0)

    loss = criterion(output_all, target_all)
    acc1 = accuracy(output_all, target_all)

    return loss.item(), acc1.item(), output_all, target_all

In [11]:
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

# train

In [12]:
sequence_path = 'D:/data/ZSX/Protein_meta_learning/uniprot/featrures/'
save_path = 'D:/data/ZSX/Protein_meta_learning/uniprot/model/'
folders = st.listdir(sequence_path)

batch_size = 32
data_seed = 42

best_params = {'learning_rate': 0.00010636954006135154,
 'weight_decay': 1.0649236744357727e-06,
 'num_experts': 9}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [13]:
for folder in folders:
    sequence_path_use = sequence_path + folder + '/'
    pos_data = st.read_file(sequence_path_use + 'Pos_model_feature.txt', index_col=0, header=None)
    neg_data = st.read_file(sequence_path_use + 'Neg_model_feature.txt', index_col=0, header=None)
    data = pd.concat([pos_data, neg_data])
    y_da = np.array([1.0] * pos_data.shape[0] + [0.0] * neg_data.shape[0])

    train_data, val_data, test_data = get_dataset(data, y_da,
                                                  batch_size=batch_size,
                                                  shuffle=True, drop_last=False,
                                                  balance=True,
                                                  train_val_test=(0.8, 0.1, 0.1),
                                                  seed=data_seed)

    save_path_model = save_path + folder + '/'
    st.mkdir(save_path_model)

    seed = 42
    set_seed(seed)

    # 定义超参数
    epochs = 30

    # lr = 3e-3
    lr = best_params['learning_rate']

    dropout_rat = 0.2
    valid_every = 1
    save_every = 5

    # 模型参数
    input_dim = 1280  # 输入维度
    hidden_dim = 256  # 门控网络的隐藏层维度
    output_dim = 2  # 输出维度（二分类任务，使用sigmoid输出）
    num_experts = best_params['num_experts']  # 专家的数量
    expert_hidden_dim = 100  # 专家网络的隐藏层维度

    # 创建 Mixture of Experts 模型
    model = MixtureOfExperts(CSA2, input_dim, hidden_dim, output_dim, num_experts, expert_hidden_dim, 
                             output_dim, dropout_rat)
    model = model.to(device)

    # 损失函数和优化器
    criterion = torch.nn.CrossEntropyLoss() # 使用 BCEWithLogitsLoss 来进行二分类
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=best_params['weight_decay'])

    train_loss = []
    train_acc = []
    val_loss = []
    val_acc = []

    # 开始训练
    for epoch in range(1, epochs + 1):
        # 执行元更新
        loss, acc = train(epoch, epochs, train_data, model, criterion, optimizer, device, log_file=None)
        print(f"[train]: Epoch {epoch} / {epochs}, Meta Loss: {loss:.4f}, Meta Acc: {acc:.4f}")

#         raise ValueError('stop')
        
        train_loss += [loss]
        train_acc += [acc]

        if epoch % valid_every == 0:
            if val_data is not None:
                val_meta_loss = 0.0
                val_meta_acc = 0.0

                loss, acc, output_all, target_all = evaluate(val_data, model, criterion, device)
                val_meta_loss += loss
                val_meta_acc += acc

                print(f"[valid]: Epoch {epoch} / {epochs}, Meta Loss: {val_meta_loss:.4f}, Meta Acc: {val_meta_acc:.4f}")

                val_loss += [val_meta_loss]
                val_acc += [val_meta_acc]

        if epoch % save_every == 0:
            optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
            save_ckpt(epoch, model, optimizer, None, 'MOE_' + folder, save_path_model)

    #     break
    print("训练完成！")

    train_info = pd.DataFrame([train_loss, val_loss, train_acc, val_acc], index=['train_loss', 'val_loss', 'train_acc', 'val_acc']).T
    st.write_file(save_path_model + 'MOE_' + folder + '_train_info.txt', train_info, index=False)

In [14]:
for folder in folders:
    save_path_model = save_path + folder + '/'
    train_info = st.read_file(save_path_model + 'MOE_' + folder + '_train_info.txt')
    train_info.index += 1
    use_model = str(train_info.iloc[4::5].sort_values('val_acc', ascending=False).index[0])
    model_path = save_path_model + 'MOE_' + folder + '_' + use_model + '.pth'
    
    shutil.copy(model_path, save_path_model + 'MOE_' + folder + '-best_model.pth')

In [15]:
DC = st.DecentColor()

plt.figure(figsize=(18, 12))

for i, folder in enumerate(folders):
    save_path_model = save_path + folder + '/'
    train_info = st.read_file(save_path_model + 'MOE_' + folder + '_train_info.txt')
    train_info.index += 1
    use_model = train_info.iloc[4::5].sort_values('val_acc', ascending=False).index[0]


    plt.subplot(3, 3, i + 1)

    plt.plot(train_info.loc[:, 'train_loss'], color=DC.c_blue)
    plt.plot(train_info.loc[:, 'val_loss'], color=DC.c_red)

    plt.scatter([use_model, use_model], train_info.loc[use_model].values[:2], c=[DC.c_blue, DC.c_red])
    plt.title(folder)
    
plt.savefig(save_path + 'loss_line.png', dpi=600, bbox_inches='tight', transparent=True)
plt.close()

# test

In [16]:
def get_precision_recall_curve(output_all_, target_all_, resolution=100, rounding=3):
    result_size = len(output_all_)
    stride = int(np.ceil(result_size / resolution))
    target_info_ = st.count_list(target_all_, dict_out=True)

    predict = pd.DataFrame(output_all_)
    predict.loc[:, 'label'] = target_all_
    predict = predict.sort_values(0)

    # precision and recall
    recall_list_ = [np.sum(predict.iloc[:i, -1]) / target_info_[1] for i in range(1, result_size + 1, stride)]
    recall_info = np.array([recall_list_[i + 1] - recall_list_[i] for i in range(len(recall_list_) - 1)])
    precision_list_ = [np.sum(predict.iloc[:i, -1]) / i for i in range(1, result_size + 1, stride)]
    precision_info = np.array([(precision_list_[i + 1] + precision_list_[i]) / 2
                               for i in range(len(precision_list_) - 1)])

    average_precision_ = np.round(np.dot(precision_info, recall_info), rounding)

    # different threshold
    recall_list2_ = []
    precision_list2_ = []
    accuracy_list2_ = []

    prob_list_ = []
    for thres in [i / resolution for i in range(1, 1 + resolution)]:
        num = np.argmin(predict.iloc[:, 1] >= thres)
        tp = np.sum(predict.iloc[:num, -1] == 1)
        tn = np.sum(predict.iloc[num:, -1] == 0)
        recall_list2_ += [tp / target_info_[1]]
        precision_list2_ += [tp / num]
        accuracy_list2_ += [(tn + tp) / predict.shape[0]]

        probs = np.array(predict.iloc[:, 1])
        probs = probs[probs >= (thres - (1 / resolution))]
        probs = probs[probs < thres]
        prob_list_ += [len(probs) / predict.shape[0]]

    return (recall_list_, precision_list_, average_precision_), \
           (recall_list2_, precision_list2_, accuracy_list2_, prob_list_)

In [17]:
def metrics_plot(output_all_, target_all_, plot_path=True, resolution=100, rounding=3):
    target_all_ = target_all_.cpu().detach().numpy()
    output_all_ = output_all_.cpu().detach().numpy()

    predict_all_ = np.argmax(output_all_, axis=1)
    conf_mat = confusion_matrix(predict_all_, target_all_)
    precision_ = conf_mat[1, 1] / sum(conf_mat[1, :])
    recall_ = conf_mat[1, 1] / sum(conf_mat[:, 1])

    fpr, tpr, threshold = metrics.roc_curve(target_all_, output_all_[:, 1])
    roc_auc_ = metrics.auc(fpr, tpr)  # 计算auc的值

    fig2_info, fig3_info = get_precision_recall_curve(output_all_, target_all_,
                                                      resolution=resolution, rounding=rounding)
    recall_list, precision_list, average_precision_ = fig2_info
    recall_list2_, precision_list2_, accuracy_list2_, prob_list_ = fig3_info

    if not plot_path:
        return precision_, recall_, roc_auc_, average_precision_

    plt.figure(figsize=(18, 4))
    lw = 2

    plt.subplot(131)
    plt.plot(fpr, tpr, color='darkorange',
             lw=lw, label='ROC curve (area = %0.3f)' % roc_auc_)
    plt.plot([0, 1], [0, 1], color='navy', lw=lw, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.legend(loc="lower right")

    plt.subplot(132)
    plt.fill_between(recall_list, precision_list, [0] * len(precision_list), color=np.array([204, 204, 228]) / 255,
                     label='Average precision:' + str(average_precision_))
    plt.xlim(0, 1)
    plt.ylim(0, 1.05)
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.legend(loc="lower right")

    plt.subplot(133)
    plt.plot(np.array(list(range(resolution))) / resolution, recall_list2_,
             lw=lw,
             color=np.array([70, 169, 224]) / 255,  # blue
             label='Recall')
    plt.plot(np.array(list(range(resolution))) / resolution, precision_list2_,
             lw=lw,
             color=np.array([181, 170, 210]) / 255,  # purple
             label='Precision')
    plt.plot(np.array(list(range(resolution))) / resolution, accuracy_list2_,
             lw=lw,
             color=np.array([239, 92, 85]) / 255,  # red
             label='Accuracy')
    plt.plot(np.array(list(range(resolution))) / resolution, prob_list_,
             lw=lw,
             color=np.array([163, 218, 202]) / 255,  # green
             label='Prob distribution')

    plt.xlim(0, 1)
    plt.ylim(0, 1.05)
    plt.xlabel('Threshold')
    plt.ylabel('Metrics')
    plt.legend(loc="lower right")

    plt.savefig(plot_path, bbox_inches='tight', dpi=600, transparent=True)
    plt.close()

    return precision_, recall_, roc_auc_, average_precision_

In [18]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve, CalibrationDisplay
from sklearn.utils.validation import check_array, check_consistent_length
from sklearn.metrics import brier_score_loss
from scipy.interpolate import interp1d
import warnings
warnings.filterwarnings('ignore')

def calculate_calibration_curve_with_ci(y_true, y_prob, n_bins=5, strategy='quantile',
                                        n_bootstrap=1000, confidence_level=0.95):
    """
    计算校准曲线并包含置信区间

    参数:
    y_true: 真实标签 (0或1)
    y_prob: 预测概率 (0到1)
    n_bins: 分桶数量
    strategy: 分桶策略 ('equal', 'quantile', 'uniform')
    n_bootstrap: Bootstrap重复次数
    confidence_level: 置信水平 (0.95表示95%置信区间)

    返回:
    结果字典，包含校准曲线、置信区间和Brier分数等信息
    """
    # 基础校准曲线
    prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=n_bins, strategy=strategy)

    # 计算Brier分数
    brier_score = np.mean((y_prob - y_true) ** 2)

    # 计算Brier分数的标准误差（通过Bootstrap）
    brier_scores_bootstrap = []
    n_samples = len(y_true)

    for i in range(n_bootstrap):
        # Bootstrap采样
        indices = np.random.choice(n_samples, n_samples, replace=True)
        y_true_boot = y_true[indices]
        y_prob_boot = y_prob[indices]

        # 计算Brier分数
        brier_boot = np.mean((y_prob_boot - y_true_boot) ** 2)
        brier_scores_bootstrap.append(brier_boot)

    brier_scores_bootstrap = np.array(brier_scores_bootstrap)
    brier_mean = np.mean(brier_scores_bootstrap)
    brier_std = np.std(brier_scores_bootstrap)

    # 计算置信区间（百分位数法）
    alpha = 1 - confidence_level
    brier_ci_lower = np.percentile(brier_scores_bootstrap, alpha/2 * 100)
    brier_ci_upper = np.percentile(brier_scores_bootstrap, (1 - alpha/2) * 100)

    # 计算校准曲线的置信区间（通过Bootstrap）
    n_bins_actual = len(prob_true)
    prob_true_ci_lower = np.zeros(n_bins_actual)
    prob_true_ci_upper = np.zeros(n_bins_actual)
    prob_pred_ci_lower = np.zeros(n_bins_actual)
    prob_pred_ci_upper = np.zeros(n_bins_actual)

    # 为每个桶计算Bootstrap置信区间
    for i in range(n_bins_actual):
        prob_true_boot_list = []
        prob_pred_boot_list = []

        for _ in range(n_bootstrap):
            # Bootstrap采样
            indices = np.random.choice(n_samples, n_samples, replace=True)
            y_true_boot = y_true[indices]
            y_prob_boot = y_prob[indices]

            # 计算校准曲线
            prob_true_boot, prob_pred_boot = calibration_curve(
                y_true_boot, y_prob_boot, n_bins=n_bins, strategy=strategy
            )

            # 如果Bootstrap结果中包含该桶
            if i < len(prob_true_boot):
                prob_true_boot_list.append(prob_true_boot[i])
                prob_pred_boot_list.append(prob_pred_boot[i])

        if len(prob_true_boot_list) > 0:
            prob_true_ci_lower[i] = np.percentile(prob_true_boot_list, alpha/2 * 100)
            prob_true_ci_upper[i] = np.percentile(prob_true_boot_list, (1 - alpha/2) * 100)
            prob_pred_ci_lower[i] = np.percentile(prob_pred_boot_list, alpha/2 * 100)
            prob_pred_ci_upper[i] = np.percentile(prob_pred_boot_list, (1 - alpha/2) * 100)

    # 计算校准误差
    calibration_error = np.mean((prob_true - prob_pred) ** 2)

    # 计算区分度（Discrimination）
    discrimination = brier_score - calibration_error

    return {
        'prob_true': prob_true,
        'prob_pred': prob_pred,
        'brier_score': brier_score,
        'brier_mean': brier_mean,
        'brier_std': brier_std,
        'brier_ci_lower': brier_ci_lower,
        'brier_ci_upper': brier_ci_upper,
        'prob_true_ci_lower': prob_true_ci_lower,
        'prob_true_ci_upper': prob_true_ci_upper,
        'prob_pred_ci_lower': prob_pred_ci_lower,
        'prob_pred_ci_upper': prob_pred_ci_upper,
        'calibration_error': calibration_error,
        'discrimination': discrimination,
        'n_samples': len(y_true),
        'n_positive': y_true.sum(),
        'n_negative': len(y_true) - y_true.sum()
    }

def plot_calibration_curve_with_ci(results, model_name="Model", ax=None,
                                   show_brier=True, show_ci=True, show_smooth=False):
    """
    绘制带置信区间的校准曲线

    参数:
    results: 计算结果字典
    model_name: 模型名称
    ax: 绘图坐标轴
    show_brier: 是否显示Brier分数
    show_ci: 是否显示置信区间
    show_smooth: 是否显示平滑曲线
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 8))
    else:
        fig = ax.figure

    prob_true = results['prob_true']
    prob_pred = results['prob_pred']

    # 1. 绘制完美校准线
    ax.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Perfect Calibration Line')

    # 2. 绘制校准点
    scatter = ax.scatter(prob_pred, prob_true, color='blue', s=80, alpha=0.8,
                         label=f'{model_name} Calibration scatter')

    # 3. 绘制平滑曲线，使用线性插值
    f = interp1d(prob_pred, prob_true, kind='linear', fill_value='extrapolate')
    x_smooth = np.linspace(0, 1, 200)
    y_smooth = f(x_smooth)
    ax.plot(x_smooth, y_smooth, 'r-', alpha=0.7, linewidth=2,
            label='Smooth Calibration Curve')

    # 4. 绘制置信区间（误差条）
    if show_ci:
        # 计算每个点的误差范围
        prob_true_error_lower = prob_true - results['prob_true_ci_lower']
        prob_true_error_upper = results['prob_true_ci_upper'] - prob_true

        # 使用errorbar绘制置信区间
        ax.errorbar(prob_pred, prob_true,
                    yerr=[prob_true_error_lower, prob_true_error_upper],
                    fmt='none', ecolor='gray', elinewidth=2, capsize=5,
                    alpha=0.6, label='95% Confidence Interval')

        # 绘制预测概率的置信区间（可选）
        ax.errorbar(prob_pred, prob_true,
                    xerr=[prob_pred - results['prob_pred_ci_lower'],
                          results['prob_pred_ci_upper'] - prob_pred],
                    fmt='none', ecolor='lightgray', elinewidth=1.5, capsize=3,
                    alpha=0.4)

    # 5. 添加统计信息文本框
    if show_brier:
        text_str = (
            f"Brier Score: {results['brier_score']:.4f}\n"
            f"95% CI: [{results['brier_ci_lower']:.4f}, {results['brier_ci_upper']:.4f}]\n"
            f"Calibration Error: {results['calibration_error']:.4f}\n"
            f"Discrimination: {results['discrimination']:.4f}\n"
            f"N samples: {results['n_samples']}\n"
            f"N Positive: {results['n_positive']} ({results['n_positive']/results['n_samples']*100:.1f}%)"
        )

        ax.text(0.02, 0.98, text_str, transform=ax.transAxes,
                fontsize=10, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

    # 6. 设置图形属性
    ax.set_xlabel('Predict Prob', fontsize=12)
    ax.set_ylabel('True Positive Ratio', fontsize=12)
    ax.set_title(f'{model_name} Calibration Curve', fontsize=14)
    ax.set_xlim(-0.09, 1.09)
    ax.set_ylim(-0.09, 1.09)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='lower right')

    return fig, ax

def plot_brier_analysis(results, model_name="Model"):
    """
    绘制Brier分数的详细分析图
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # 1. Brier分数分布图（Bootstrap）
    ax1 = axes[0]
    # 这里我们无法直接获取Bootstrap分布，但可以显示Brier分数
    brier_score = results['brier_score']
    brier_ci_lower = results['brier_ci_lower']
    brier_ci_upper = results['brier_ci_upper']

    # 绘制Brier分数和置信区间
    ax1.barh(0, brier_score, height=0.4, color='skyblue', alpha=0.7,
             label=f'Brier Score = {brier_score:.4f}')

    # 绘制置信区间
    ax1.errorbar(brier_score, 0,
                 xerr=[[brier_score - brier_ci_lower], [brier_ci_upper - brier_score]],
                 fmt='none', ecolor='red', elinewidth=2, capsize=5)

    ax1.set_yticks([])
    ax1.set_xlabel('Brier Score', fontsize=12)
    ax1.set_title(f'{model_name} Brier Score Analyze', fontsize=14)
    ax1.set_xlim(0, 1)
    ax1.grid(True, alpha=0.3, axis='x')
    ax1.legend()

    # 2. Brier分数分解（校准度 vs 区分度）
    ax2 = axes[1]
    components = ['Calibration', 'Discrimination']
    values = [results['calibration_error'], results['discrimination']]
    colors = ['lightcoral', 'lightgreen']

    bars = ax2.bar(components, values, color=colors, alpha=0.7)

    # 在柱子上方添加数值
    for bar, value in zip(bars, values):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                 f'{value:.4f}', ha='center', va='bottom', fontsize=11)

    ax2.set_ylabel('Contribution value', fontsize=12)
    ax2.set_title('Brier score decomposition', fontsize=14)
    ax2.set_ylim(0, max(values) * 1.2)
    ax2.grid(True, alpha=0.3, axis='y')

    # 添加总Brier分数
    ax2.text(0.5, max(values) * 1.15,
             f'Overall Brier score: {brier_score:.4f}',
             ha='center', fontsize=12, fontweight='bold')

    plt.tight_layout()
    plt.show()

def comprehensive_calibration_analysis(y_true, y_prob, model_name="Model",
                                       n_bins=10, n_bootstrap=1000, show_smooth=False):
    """
    综合校准分析：包含校准曲线、置信区间、Brier分数和详细分析

    参数:
    y_true: 真实标签
    y_prob: 预测概率
    model_name: 模型名称
    n_bins: 分桶数
    n_bootstrap: Bootstrap次数

    返回:
    结果字典
    """
    # 1. 计算校准曲线和统计量
    results = calculate_calibration_curve_with_ci(
        y_true, y_prob, n_bins=n_bins, n_bootstrap=n_bootstrap
    )

    # 2. 绘制带置信区间的校准曲线
    fig1, ax1 = plt.subplots(figsize=(10, 8))
    plot_calibration_curve_with_ci(results, model_name=model_name, ax=ax1, show_smooth=show_smooth)
    plt.tight_layout()
    plt.show()

    # 3. 绘制Brier分数分析
    plot_brier_analysis(results, model_name=model_name)

    return results

In [19]:
plt.figure(figsize=(15, 15))
i = 0
for folder in folders:
    i += 1
    sequence_path_use = sequence_path + folder + '/'
    pos_data = st.read_file(sequence_path_use + 'Pos_model_feature.txt', index_col=0, header=None)
    neg_data = st.read_file(sequence_path_use + 'Neg_model_feature.txt', index_col=0, header=None)
    data = pd.concat([pos_data, neg_data])
    y_da = np.array([1.0] * pos_data.shape[0] + [0.0] * neg_data.shape[0])

    train_data, val_data, test_data = get_dataset(data, y_da,
                                                  batch_size=batch_size,
                                                  shuffle=True, drop_last=False,
                                                  balance=True,
                                                  train_val_test=(0.8, 0.1, 0.1),
                                                  seed=data_seed)
    seed = 42
    set_seed(seed)

    lr = best_params['learning_rate']
    dropout_rat = 0.2

    # 模型参数
    input_dim = 1280  # 输入维度
    hidden_dim = 256  # 门控网络的隐藏层维度
    output_dim = 2  # 输出维度（二分类任务，使用sigmoid输出）
    num_experts = best_params['num_experts']  # 专家的数量
    expert_hidden_dim = 100  # 专家网络的隐藏层维度

    # 创建 Mixture of Experts 模型
    model = MixtureOfExperts(CSA2, input_dim, hidden_dim, output_dim, num_experts, expert_hidden_dim, 
                             output_dim, dropout_rat)
    model = model.to(device)

    # 损失函数和优化器
    criterion = torch.nn.CrossEntropyLoss() # 使用 BCEWithLogitsLoss 来进行二分类
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=best_params['weight_decay'])

    set_seed(seed)
    ckpt_file_use = 'D:/data/ZSX/Protein_meta_learning/uniprot/model/MOE_20250416_' + folder + '/MOE_' + folder + '-best_model.pth'
    checkpoint = torch.load(ckpt_file_use, weights_only=False)
    pa_dict = checkpoint['model_state_dict']
    model.load_state_dict(pa_dict)

    set_seed(seed)
    model.eval()

    val_loss, val_acc, output_val, target_val = evaluate(val_data, model, criterion, device)
    test_loss, test_acc, output_test, target_test = evaluate(test_data, model, criterion, device)
    
    y_true = np.array(target_test.cpu())
    y_prob = np.array(output_test.detach().cpu())[:, 1]
    
    plot_path_fig = 'D:/设计/protein_evaluation/revision/model_performance/Interval_' + folder + '_calibration.png'
    plot_path_fig2 = 'D:/设计/protein_evaluation/revision/model_performance/Simple_' + folder + '_calibration.png'
    plot_path_info = 'D:/设计/protein_evaluation/revision/model_performance/Table_' + folder + '_metrics.txt'
    
    n_bins=5
    n_bootstrap=1000

    results = calculate_calibration_curve_with_ci(y_true, y_prob, n_bins=n_bins, n_bootstrap=n_bootstrap)
    result_df = pd.DataFrame([[key] + list(value) if type(value) == np.ndarray else [key, value] for key, value in results.items()])
    
    # 绘制带置信区间的校准曲线
    ax1 = plt.subplot(3, 3, i)
    plot_calibration_curve_with_ci(results, model_name=folder, ax=ax1)

plt.savefig('D:/设计/protein_evaluation/revision/model_performance/Interval_All_calibration.png', 
            dpi=600, bbox_inches='tight', transparent=True)
plt.close()

In [21]:
def get_precision_list(output_all_, target_all_, resolution=100, rounding=3):
    target_all_ = target_all_.cpu().detach().numpy()
    output_all_ = output_all_.cpu().detach().numpy()
    
    result_size = len(output_all_)
    stride = int(np.ceil(result_size / resolution))
    target_info_ = st.count_list(target_all_, dict_out=True)

    predict = pd.DataFrame(output_all_)
    predict.loc[:, 'label'] = target_all_
    predict = predict.sort_values(0)

    # different threshold
    recall_list2_ = []
    precision_list2_ = []
    accuracy_list2_ = []

    prob_list_ = []
    for thres in [i / resolution for i in range(1, 1 + resolution)]:
        num = np.argmin(predict.iloc[:, 1] >= thres)
        tp = np.sum(predict.iloc[:num, -1] == 1)
        tn = np.sum(predict.iloc[num:, -1] == 0)
        recall_list2_ += [tp / target_info_[1]]
        precision_list2_ += [tp / num]
        accuracy_list2_ += [(tn + tp) / predict.shape[0]]

        probs = np.array(predict.iloc[:, 1])
        probs = probs[probs >= (thres - (1 / resolution))]
        probs = probs[probs < thres]
        prob_list_ += [len(probs) / predict.shape[0]]

    return recall_list2_, precision_list2_, accuracy_list2_, prob_list_

In [22]:
recall_list2_test_all = []
precision_list2_test_all = []
accuracy_list2_test_all = []
prob_list_test_all = []
for folder in folders:
    sequence_path_use = sequence_path + folder + '/'
    pos_data = st.read_file(sequence_path_use + 'Pos_model_feature.txt', index_col=0, header=None)
    neg_data = st.read_file(sequence_path_use + 'Neg_model_feature.txt', index_col=0, header=None)
    data = pd.concat([pos_data, neg_data])
    y_da = np.array([1.0] * pos_data.shape[0] + [0.0] * neg_data.shape[0])

    train_data, val_data, test_data = get_dataset(data, y_da,
                                                  batch_size=batch_size,
                                                  shuffle=True, drop_last=False,
                                                  balance=True,
                                                  train_val_test=(0.8, 0.1, 0.1),
                                                  seed=data_seed)
    seed = 42
    set_seed(seed)

    lr = best_params['learning_rate']
    dropout_rat = 0.2

    # 模型参数
    input_dim = 1280  # 输入维度
    hidden_dim = 256  # 门控网络的隐藏层维度
    output_dim = 2  # 输出维度（二分类任务，使用sigmoid输出）
    num_experts = best_params['num_experts']  # 专家的数量
    expert_hidden_dim = 100  # 专家网络的隐藏层维度

    # 创建 Mixture of Experts 模型
    model = MixtureOfExperts(CSA2, input_dim, hidden_dim, output_dim, num_experts, expert_hidden_dim, 
                             output_dim, dropout_rat)
    model = model.to(device)

    # 损失函数和优化器
    criterion = torch.nn.CrossEntropyLoss() # 使用 BCEWithLogitsLoss 来进行二分类
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=best_params['weight_decay'])

    set_seed(seed)
    ckpt_file_use = save_path + folder + '/MOE_' + folder + '-best_model.pth'
    checkpoint = torch.load(ckpt_file_use, weights_only=False)
    pa_dict = checkpoint['model_state_dict']
    model.load_state_dict(pa_dict)

    set_seed(seed)
    model.eval()

    val_loss, val_acc, output_val, target_val = evaluate(val_data, model, criterion, device)
    test_loss, test_acc, output_test, target_test = evaluate(test_data, model, criterion, device)

    
    recall_list2_val, precision_list2_val, accuracy_list2_val, prob_list_val = get_precision_list(output_val, target_val, 
                                                                                                  resolution=1000, rounding=4)
    recall_list2_test, precision_list2_test, accuracy_list2_test, prob_list_test = get_precision_list(output_test, target_test, 
                                                                                                      resolution=1000, rounding=4)
    
    recall_list2_test_all += [recall_list2_test]
    precision_list2_test_all += [precision_list2_test]
    accuracy_list2_test_all += [accuracy_list2_test]
    prob_list_test_all += [prob_list_test]

In [23]:
resolution = 1000
recall_list2_test_df = pd.DataFrame(recall_list2_test_all, index=folders, columns=[i / resolution for i in range(1, 1 + resolution)])
precision_list2_test_df = pd.DataFrame(precision_list2_test_all, index=folders, columns=[i / resolution for i in range(1, 1 + resolution)])
accuracy_list2_test_df = pd.DataFrame(accuracy_list2_test_all, index=folders, columns=[i / resolution for i in range(1, 1 + resolution)])
prob_list_test_df = pd.DataFrame(prob_list_test_all, index=folders, columns=[i / resolution for i in range(1, 1 + resolution)])

st.write_file('D:/设计/protein_evaluation/ana/predict_uniprot_unreviewed/recall_test.txt', recall_list2_test_df.T)
st.write_file('D:/设计/protein_evaluation/ana/predict_uniprot_unreviewed/precision_test.txt', precision_list2_test_df.T)
st.write_file('D:/设计/protein_evaluation/ana/predict_uniprot_unreviewed/accuracy_test.txt', accuracy_list2_test_df.T)
st.write_file('D:/设计/protein_evaluation/ana/predict_uniprot_unreviewed/prob_test.txt', prob_list_test_df.T)